In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Impostazioni predefinite e opzioni di configurazione della transpilazione


<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice presente in questa pagina è stato sviluppato con i seguenti requisiti.
Si consiglia di utilizzare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
I circuiti astratti devono essere transpilati perché le QPU dispongono di un insieme limitato di gate di base e non possono eseguire operazioni arbitrarie. La funzione del transpilatore è trasformare circuiti arbitrari in modo che possano essere eseguiti su una QPU specificata. Questo avviene traducendo i circuiti nei gate di base supportati e introducendo gate SWAP dove necessario, affinché la connettività del circuito corrisponda a quella della QPU.

Come spiegato in [Transpilare con i pass manager](transpile-with-pass-managers), puoi creare un [pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) usando la funzione [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) e passare un circuito o una lista di circuiti al suo metodo [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) per transpiliarli. Puoi chiamare `generate_preset_pass_manager` specificando solo il livello di ottimizzazione e il backend, scegliendo di utilizzare i valori predefiniti per tutte le altre opzioni, oppure puoi passare argomenti aggiuntivi alla funzione per ottimizzare la transpilazione.

## Utilizzo di base senza parametri
In questo esempio, passiamo un circuito e la QPU di destinazione al transpilatore senza specificare ulteriori parametri.

Crea un circuito e visualizza il risultato:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

Transpila il circuito e visualizza il risultato:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## Tutti i parametri disponibili
Di seguito sono elencati tutti i parametri disponibili per la funzione [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). Esistono due categorie di argomenti: quelli che descrivono la destinazione della compilazione e quelli che influenzano il funzionamento del transpilatore.

Tutti i parametri tranne `optimization_level` sono opzionali. Per i dettagli completi, consulta la [documentazione API del Transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) - Quanta ottimizzazione eseguire sui circuiti. Intero nell'intervallo (0 - 3). Livelli più alti generano circuiti più ottimizzati, a scapito di tempi di transpilazione più lunghi. Vedi [Impostare il livello di ottimizzazione del transpilatore](set-optimization) per ulteriori dettagli.

### Parametri usati per descrivere la destinazione di compilazione:
Questi argomenti descrivono la QPU di destinazione per l'esecuzione del circuito, incluse informazioni come la coupling map della QPU (che descrive la connettività dei qubit), i gate di base supportati dalla QPU e i tassi di errore dei gate.

Molti di questi parametri sono descritti in dettaglio in [Parametri comunemente usati per la transpilazione](common-parameters).

<details>
  <summary>
    **Parametri QPU (`Backend`)**
  </summary>

**Parametri Backend** - Se specifichi `backend`, non è necessario specificare `target` o altre opzioni del backend. Allo stesso modo, se specifichi `target`, non è necessario specificare `backend` o altre opzioni del backend.
- `backend` (Backend) - Se impostato, il transpilatore compila il circuito di input per questo dispositivo. Se viene impostata un'altra opzione che influisce su queste impostazioni, come `coupling_map`, tale opzione sovrascrive le impostazioni di `backend`.
- `target` (Target) - Un target del transpilatore del backend. Normalmente è specificato come parte dell'argomento backend, ma se hai costruito manualmente un oggetto Target puoi specificarlo qui. Questo sovrascrive il target di `backend`.
- `backend_properties` (BackendProperties) - Proprietà restituite da una QPU, incluse informazioni sugli errori dei gate, gli errori di readout, i tempi di coerenza dei qubit e così via. Trova una QPU che fornisce queste informazioni eseguendo `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) - Una restrizione opzionale dell'hardware di controllo sulla risoluzione temporale delle istruzioni. Queste informazioni sono fornite dalla configurazione della QPU. Se la QPU non ha restrizioni sull'allocazione dei tempi delle istruzioni, `timing_constraints` è `None` e non viene eseguita alcuna modifica. Una QPU potrebbe riportare un insieme di restrizioni, ovvero:
    - `granularity`: Un valore intero che rappresenta la risoluzione minima del gate a impulsi in unità di dt. Un gate a impulsi definito dall'utente dovrebbe avere una durata multipla di questo valore di granularità.
    - `min_length`: Un valore intero che rappresenta la lunghezza minima del gate a impulsi in unità di dt. Un gate a impulsi definito dall'utente dovrebbe essere più lungo di questa lunghezza.
    - `pulse_alignment`: Un valore intero che rappresenta la risoluzione temporale del tempo di avvio delle istruzioni di gate. Le istruzioni di gate dovrebbero iniziare a un tempo multiplo di questo valore.
    - `acquire_alignment`: Un valore intero che rappresenta la risoluzione temporale del tempo di avvio delle istruzioni di misura. Le istruzioni di misura dovrebbero iniziare a un tempo multiplo di questo valore.
</details>

<details>
  <summary>
    **Parametri di layout e topologia**
  </summary>

- `basis_gates` (List[str] | None) - Lista dei nomi dei gate di base a cui decomporre. Ad esempio ['u1', 'u2', 'u3', 'cx']. Se `None`, non viene eseguita alcuna decomposizione.
- `coupling_map` (CouplingMap | List[List[int]]) - Coupling map orientata (eventualmente personalizzata) da usare come destinazione nel mapping. Se la coupling map è simmetrica, devono essere specificate entrambe le direzioni. Sono supportati i seguenti formati:
    - Istanza di CouplingMap
    - List - deve essere fornita come matrice di adiacenza, dove ogni voce specifica tutte le interazioni dirette a due qubit supportate dalla QPU. Ad esempio: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - Mapping delle operazioni del circuito agli schedule a impulsi. Se `None`, viene usato l'`instruction_schedule_map` della QPU.
</details>

### Parametri usati per influenzare il funzionamento del transpilatore
Questi parametri influenzano specifiche fasi della transpilazione. Alcuni di essi potrebbero influenzare più fasi, ma sono elencati sotto una sola fase per semplicità. Se specifichi un argomento, come `initial_layout` per i qubit che vuoi usare, quel valore sovrascrive tutti i pass che potrebbero modificarlo. In altre parole, il transpilatore non modificherà nulla che tu abbia specificato manualmente. Per i dettagli sulle singole fasi, vedi [Fasi del transpilatore](transpiler-stages).

<details>
  <summary>
    **Fase di inizializzazione**
  </summary>

- `hls_config` (HLSConfig) - Una classe di configurazione opzionale `HLSConfig` che viene passata direttamente al pass di trasformazione `HighLevelSynthesis`. Questa classe di configurazione ti permette di specificare le liste di algoritmi di sintesi e i loro parametri per vari oggetti di alto livello.
- `init_method` (str) - Il nome del plugin da usare per la fase di inizializzazione. Per impostazione predefinita, non viene usato un plugin esterno. Puoi vedere la lista dei plugin installati eseguendo `list_stage_plugins()` con `init` come argomento del nome della fase.
- `unitary_synthesis_method` (str) - Il nome del metodo di sintesi unitaria da usare. Per impostazione predefinita viene usato `default`. Puoi vedere la lista dei plugin installati eseguendo `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) - Un dizionario di configurazione opzionale che viene passato direttamente al plugin di sintesi unitaria. Per impostazione predefinita questa impostazione non ha effetto perché il metodo di sintesi unitaria predefinito non accetta configurazioni personalizzate. L'applicazione di una configurazione personalizzata è necessaria solo quando viene specificato un plugin di sintesi unitaria con l'argomento `unitary_synthesis`. Poiché questa è specifica per ciascun plugin di sintesi unitaria, consulta la documentazione del plugin per sapere come usare questa opzione.
</details>

<details>
  <summary>
    **Fase di layout**
  </summary>

- `initial_layout` (Layout | Dict | List) - Posizione iniziale dei qubit virtuali sui qubit fisici. Se questo layout rende il circuito compatibile con i vincoli della `coupling_map`, verrà usato. Il layout finale non è garantito essere lo stesso, poiché il transpilatore potrebbe permutare i qubit tramite SWAP o altri mezzi. Per i dettagli completi, vedi la [sezione Initial layout.](common-parameters#initial-layout)
- `layout_method` (str) - Nome del pass di selezione del layout (`default`, `dense`, `sabre` e `trivial`). Può anche essere il nome del plugin esterno da usare per la fase di layout. Puoi vedere la lista dei plugin installati eseguendo `list_stage_plugins()` con `layout` come argomento `stage_name`. Il valore predefinito è `sabre`.
</details>

<details>
  <summary>
    **Fase di routing**
  </summary>

- `routing_method` (str) - Nome del pass di routing (`basic`, `lookahead`, `default`, `sabre` o `none`). Può anche essere il nome del plugin esterno da usare per la fase di routing. Puoi vedere la lista dei plugin installati eseguendo `list_stage_plugins()` con `routing` come argomento `stage_name`. Il valore predefinito è `sabre`.
</details>

<details>
  <summary>
    **Fase di traduzione**
  </summary>

- `translation_method` (str) - Nome del pass di traduzione (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). Può anche essere il nome del plugin esterno da usare per la fase di traduzione. Puoi vedere la lista dei plugin installati eseguendo `list_stage_plugins()` con `translation` come argomento `stage_name`. Il valore predefinito è `translator`.
</details>

<details>
  <summary>
    **Fase di ottimizzazione**
  </summary>

- `approximation_degree` (float, nell'intervallo 0-1 | None) - Parametro euristico usato per l'approssimazione del circuito (1.0 = nessuna approssimazione, 0.0 = approssimazione massima). Il valore predefinito è 1.0. Specificare `None` imposta il grado di approssimazione al tasso di errore riportato. Vedi la [sezione Grado di approssimazione](common-parameters#approx-degree) per ulteriori dettagli.
- `optimization_method` (str) - Il nome del plugin da usare per la fase di ottimizzazione. Per impostazione predefinita non viene usato un plugin esterno. Puoi vedere la lista dei plugin installati eseguendo `list_stage_plugins()` con `optimization` come argomento `stage_name`.
</details>

<details>
  <summary>
    **Fase di scheduling**
  </summary>

- `scheduling_method` (str) - Nome del pass di scheduling. Può anche essere il nome del plugin esterno da usare per la fase di scheduling. Puoi vedere la lista dei plugin installati eseguendo `list_stage_plugins()` con `scheduling` come argomento `stage_name`.
  * 'as_soon_as_possible': Pianifica le istruzioni in modo greedy, il prima possibile su una risorsa qubit (alias: `asap`).
  * 'as_late_as_possible': Pianifica le istruzioni il più tardi possibile, cioè mantenendo i qubit nello stato fondamentale quando possibile (alias: `alap`). Questo è il valore predefinito.
</details>

<details>
  <summary>
    **Esecuzione del transpilatore**
  </summary>

- `seed_transpiler` (int) - Imposta i seed casuali per le parti stocastiche del transpilatore.
</details>

I seguenti valori predefiniti vengono usati se non specifichi nessuno dei parametri sopra indicati. Consulta la [pagina di riferimento API](../api/qiskit/transpiler_preset) del metodo per ulteriori informazioni:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>